# Uncertainty forecasting on a public labor-market dataset

This notebook applies the probabilistic extension of the signature-transform framework to **U.S. initial jobless claims in levels**. Uncertainty is modeled through weighted quantile regression on top of signature features and adaptive signature-kernel weights. As in notebook 3, the target remains in levels, while the model forecasts **future changes over each horizon** and then converts those forecasts back into level predictions and prediction intervals.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from marketplace_signature_forecast.config import (
    ALL_HORIZONS,
    DEFAULT_DEPTH,
    DEFAULT_N_TARGET,
    DEFAULT_WINDOW_SIZE,
    FRED_API_KEY,
)
from marketplace_signature_forecast.data_loading import (
    build_data_dictionary_from_specs,
    fetch_series_bundle,
)
from marketplace_signature_forecast.dataset_specs import (
    INITIAL_CLAIMS_END_DATE,
    INITIAL_CLAIMS_EXPERIMENT_NAME,
    INITIAL_CLAIMS_FRED_INPUTS,
    INITIAL_CLAIMS_N_VALIDATION,
    INITIAL_CLAIMS_PLOT_HORIZON,
    INITIAL_CLAIMS_RESAMPLE_FREQ,
    INITIAL_CLAIMS_START_DATE,
    INITIAL_CLAIMS_TARGET,
    INITIAL_CLAIMS_YAHOO_INPUTS,
)
from marketplace_signature_forecast.preprocessing import (
    build_model_dataset,
    prepare_standardized_arrays,
    resample_collection,
    resample_to_weekly,
    save_processed_bundle,
    train_validation_split,
)
from marketplace_signature_forecast.evaluation import run_multi_horizon_experiment
from marketplace_signature_forecast.plotting import plot_quantile_forecast_interval
from marketplace_signature_forecast.quantile_evaluation import run_multi_horizon_quantile_experiment

TARGET_COL = INITIAL_CLAIMS_TARGET["name"]
TARGET_LABEL = INITIAL_CLAIMS_TARGET["name"]
PUBLIC_EXPERIMENT_NAME = INITIAL_CLAIMS_EXPERIMENT_NAME

BASE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / f"{PUBLIC_EXPERIMENT_NAME}_uncertainty"
DATASET_DIR = BASE_OUTPUT_DIR / "dataset"
Y_ONLY_DIR = BASE_OUTPUT_DIR / "signature_y_only_quantiles"
JOINT_DIR = BASE_OUTPUT_DIR / "signature_joint_path_quantiles"
POINT_Y_ONLY_DIR = BASE_OUTPUT_DIR / "point_y_only"
POINT_JOINT_DIR = BASE_OUTPUT_DIR / "point_joint_path"
PLOT_DIR = PROJECT_ROOT / "figures" / f"{PUBLIC_EXPERIMENT_NAME}_uncertainty"

for directory in [BASE_OUTPUT_DIR, DATASET_DIR, Y_ONLY_DIR, JOINT_DIR, POINT_Y_ONLY_DIR, POINT_JOINT_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

QUANTILES = [0.1, 0.5, 0.9]
QUANTILE_ALPHA = 1e-2
WINDOW_SIZE = DEFAULT_WINDOW_SIZE
DEPTH = DEFAULT_DEPTH
N_TARGET = DEFAULT_N_TARGET
ADD_TIME = True

data_dictionary = build_data_dictionary_from_specs(
    target=INITIAL_CLAIMS_TARGET,
    fred_inputs=INITIAL_CLAIMS_FRED_INPUTS,
    yahoo_inputs=INITIAL_CLAIMS_YAHOO_INPUTS,
)
target_mask = data_dictionary["type"].eq("target (y)")
data_dictionary.loc[target_mask, "variable"] = TARGET_COL
data_dictionary.loc[target_mask, "description"] = "U.S. initial jobless claims in levels"
data_dictionary.loc[target_mask, "frequency"] = "Weekly, ending Saturday"
data_dictionary.loc[target_mask, "code"] = INITIAL_CLAIMS_TARGET["code"]
data_dictionary

## 1. Rebuild the weekly public dataset

The public experiment uses the same raw FRED series as notebook 2. The target remains in **levels**, and the uncertainty-forecasting pipeline models **future changes over horizon $h$** before mapping the quantile forecasts back into level-space prediction intervals.

In [ ]:
y_raw, x_raw = fetch_series_bundle(
    api_key=FRED_API_KEY,
    target=INITIAL_CLAIMS_TARGET,
    start=INITIAL_CLAIMS_START_DATE,
    end=INITIAL_CLAIMS_END_DATE,
    fred_inputs=INITIAL_CLAIMS_FRED_INPUTS,
    yahoo_inputs=INITIAL_CLAIMS_YAHOO_INPUTS,
)

y_weekly = resample_to_weekly(y_raw, freq=INITIAL_CLAIMS_RESAMPLE_FREQ)
y_target = y_weekly.rename(columns={INITIAL_CLAIMS_TARGET["name"]: TARGET_COL})

x_weekly = resample_collection(x_raw, freq=INITIAL_CLAIMS_RESAMPLE_FREQ)
full_data = build_model_dataset(y_target, x_weekly).dropna().copy()
train_data, validation_data = train_validation_split(full_data, INITIAL_CLAIMS_N_VALIDATION)

save_processed_bundle(full_data, train_data, validation_data, data_dictionary, DATASET_DIR)

print(full_data.shape)
print(full_data.index.min(), full_data.index.max())
full_data.head()

## 2. Standardized arrays

In [ ]:
prepared = prepare_standardized_arrays(
    full_data=full_data,
    target_col=TARGET_COL,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
)

X = prepared["X"]
y = prepared["y"]
dates = prepared["dates"]
y_scaler = prepared["y_scaler"]

print(X.shape, y.shape)

## 3. Probabilistic forecast: target-path signature only

In [ ]:
quantile_results_y_only, quantile_summary_y_only = run_multi_horizon_quantile_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    quantiles=QUANTILES,
    output_dir=Y_ONLY_DIR,
    alpha=QUANTILE_ALPHA,
    n_target=N_TARGET,
    use_sig_y_only=True,
    add_time=ADD_TIME,
)

quantile_summary_y_only = quantile_summary_y_only.rename(
    columns={
        "pinball_q0_1": "pinball_q0_1_y_only",
        "pinball_q0_5": "pinball_q0_5_y_only",
        "pinball_q0_9": "pinball_q0_9_y_only",
        "median_mae": "median_mae_y_only",
        "median_rmse": "median_rmse_y_only",
        "median_mre": "median_mre_y_only",
        "interval_coverage": "interval_coverage_y_only",
        "avg_interval_width": "avg_interval_width_y_only",
    }
)
quantile_summary_y_only

## 4. Probabilistic forecast: joint-path signature

In [ ]:
quantile_results_joint, quantile_summary_joint = run_multi_horizon_quantile_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    quantiles=QUANTILES,
    output_dir=JOINT_DIR,
    alpha=QUANTILE_ALPHA,
    n_target=N_TARGET,
    use_sig_y_only=False,
    add_time=ADD_TIME,
)

quantile_summary_joint = quantile_summary_joint.rename(
    columns={
        "pinball_q0_1": "pinball_q0_1_joint",
        "pinball_q0_5": "pinball_q0_5_joint",
        "pinball_q0_9": "pinball_q0_9_joint",
        "median_mae": "median_mae_joint",
        "median_rmse": "median_rmse_joint",
        "median_mre": "median_mre_joint",
        "interval_coverage": "interval_coverage_joint",
        "avg_interval_width": "avg_interval_width_joint",
    }
)
quantile_summary_joint

## 5. Compare probabilistic and point forecasts

Because both the point model and the quantile model forecast **changes in the level series**, the most informative point diagnostics are **MAE** and **RMSE**.  
The summary below therefore compares the quantile model's median forecast against the point-forecast benchmark using those scale-sensitive metrics, while interval coverage and interval width summarize the uncertainty layer.

In [ ]:
point_results_y_only, _ = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=POINT_Y_ONLY_DIR,
    use_sig_y_only=True,
    add_time=ADD_TIME,
)

point_results_joint, _ = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=POINT_JOINT_DIR,
    use_sig_y_only=False,
    add_time=ADD_TIME,
)

point_summary_y_only = pd.DataFrame(
    [
        {
            "horizon_weeks": horizon,
            "point_mae_y_only": point_results_y_only[horizon]["mae"],
            "point_rmse_y_only": point_results_y_only[horizon]["rmse"],
            "point_mre_y_only": point_results_y_only[horizon]["mre"],
            "n_forecasts": len(point_results_y_only[horizon]["t"]),
        }
        for horizon in sorted(point_results_y_only)
    ]
)

point_summary_joint = pd.DataFrame(
    [
        {
            "horizon_weeks": horizon,
            "point_mae_joint": point_results_joint[horizon]["mae"],
            "point_rmse_joint": point_results_joint[horizon]["rmse"],
            "point_mre_joint": point_results_joint[horizon]["mre"],
            "n_forecasts": len(point_results_joint[horizon]["t"]),
        }
        for horizon in sorted(point_results_joint)
    ]
)

summary_table = quantile_summary_y_only.merge(
    quantile_summary_joint,
    on=["horizon_weeks", "n_forecasts"],
    how="outer",
).merge(
    point_summary_y_only,
    on=["horizon_weeks", "n_forecasts"],
    how="left",
).merge(
    point_summary_joint,
    on=["horizon_weeks", "n_forecasts"],
    how="left",
)

summary_table.to_csv(BASE_OUTPUT_DIR / "initial_claims_uncertainty_summary.csv", index=False)
summary_table.round(4)

## 6. Prediction intervals for a representative horizon

In [ ]:
forecast_y_only = pd.read_csv(Y_ONLY_DIR / f"quantile_forecast_results_delta{INITIAL_CLAIMS_PLOT_HORIZON}.csv")
forecast_joint = pd.read_csv(JOINT_DIR / f"quantile_forecast_results_delta{INITIAL_CLAIMS_PLOT_HORIZON}.csv")

plot_quantile_forecast_interval(
    forecast_df=forecast_y_only,
    delta_t=INITIAL_CLAIMS_PLOT_HORIZON,
    target_label=TARGET_LABEL,
    path=PLOT_DIR / f"initial_claims_interval_y_only_delta{INITIAL_CLAIMS_PLOT_HORIZON}.png",
)

plot_quantile_forecast_interval(
    forecast_df=forecast_joint,
    delta_t=INITIAL_CLAIMS_PLOT_HORIZON,
    target_label=TARGET_LABEL,
    path=PLOT_DIR / f"initial_claims_interval_joint_delta{INITIAL_CLAIMS_PLOT_HORIZON}.png",
)

summary_table.loc[summary_table["horizon_weeks"] == INITIAL_CLAIMS_PLOT_HORIZON].T